In [2]:
import pandas as pd
import numpy as np
data = 'preprocessed_data(1).csv'
df = pd.read_csv(data)

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 596 entries, 0 to 595
Data columns (total 17 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Gender                   596 non-null    int64  
 1   Year                     596 non-null    int64  
 2   STEM_Major               596 non-null    int64  
 3   Diet_Quality             596 non-null    int64  
 4   Internet_Quality         596 non-null    int64  
 5   Employment               596 non-null    int64  
 6   Stress_Level             596 non-null    float64
 7   AI_Tool_Usage            596 non-null    int64  
 8   GPA                      596 non-null    float64
 9   Performance_Tier         596 non-null    int64  
 10  Total_Productive_Hrs     596 non-null    float64
 11  Distraction_Hrs          596 non-null    float64
 12  Study_Distraction_Ratio  596 non-null    float64
 13  Wellbeing_Score          596 non-null    float64
 14  Available_Study_Time     596 non-null

In [4]:
df.head(20)


,Gender,Year,STEM_Major,Diet_Quality,Internet_Quality,Employment,Stress_Level,AI_Tool_Usage,GPA,Performance_Tier,Total_Productive_Hrs,Distraction_Hrs,Study_Distraction_Ratio,Wellbeing_Score,Available_Study_Time,Active_Engagement_Score,Study_Quality
0,1,2,1,3,3,0,5.5,3,3.39,1,4.0,2.0,1.904762,5.0,14.625,11.5,4.0
1,0,1,1,4,3,0,5.5,4,3.69,1,2.0,1.5,1.250000,6.0,14.125,10.5,6.0
2,1,1,1,2,3,0,7.5,3,3.48,1,5.0,5.0,0.588235,1.0,11.875,12.5,6.0
3,1,2,1,3,3,0,5.5,4,3.50,1,2.0,4.0,0.487805,-0.5,14.625,11.2,6.0
4,1,2,1,2,2,0,7.5,4,3.70,1,4.0,5.0,0.784314,0.0,11.750,11.2,12.0
5,1,3,1,2,4,0,9.5,4,2.90,1,4.0,8.5,0.465116,0.5,10.375,9.7,12.0
6,0,2,0,2,3,0,3.5,3,3.05,1,1.0,5.0,0.196078,2.5,11.750,11.2,3.0
7,1,1,1,3,1,0,7.5,3,3.00,1,8.0,4.0,0.731707,3.5,13.875,13.5,3.0
8,1,3,0,2,2,2,1.5,2,3.50,1,3.0,2.0,1.428571,8.0,13.625,11.2,9.0
9,1,3,1,3,1,1,5.5,1,3.89,2,2.0,4.0,0.000000,-1.5,15.875,10.5,0.0


In [5]:
df.columns.tolist()

['Gender',
 'Year',
 'STEM_Major',
 'Diet_Quality',
 'Internet_Quality',
 'Employment',
 'Stress_Level',
 'AI_Tool_Usage',
 'GPA',
 'Performance_Tier',
 'Total_Productive_Hrs',
 'Distraction_Hrs',
 'Study_Distraction_Ratio',
 'Wellbeing_Score',
 'Available_Study_Time',
 'Active_Engagement_Score',
 'Study_Quality']

In [6]:
df['Diet_Quality'].unique()

array([3, 4, 2, 1])

In [7]:
from sklearn.model_selection import train_test_split
X = df.drop(['GPA', 'Performance_Tier'], axis=1)
Y = df['Performance_Tier']
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

In [8]:
from sklearn.preprocessing import StandardScaler
s = StandardScaler()
X_train_scaled = s.fit_transform(X_train)
X_test_scaled = s.transform(X_test)
print(X_train_scaled.shape)
print(X_test_scaled.shape)

(476, 15)
(120, 15)


In [9]:
from sklearn.feature_selection import SelectKBest, f_classif

selector = SelectKBest(score_func=f_classif, k=15)
X_train_selected = selector.fit_transform(X_train_scaled, Y_train)
X_test_selected  = selector.transform(X_test_scaled)

selected_cols = X_train.columns[selector.get_support()]
print("Selected features:", selected_cols.tolist())
print(X_train_selected.shape)

Selected features: ['Gender', 'Year', 'STEM_Major', 'Diet_Quality', 'Internet_Quality', 'Employment', 'Stress_Level', 'AI_Tool_Usage', 'Total_Productive_Hrs', 'Distraction_Hrs', 'Study_Distraction_Ratio', 'Wellbeing_Score', 'Available_Study_Time', 'Active_Engagement_Score', 'Study_Quality']
(476, 15)


In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier , GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

models = {
    'Lr': LogisticRegression(
        class_weight='balanced', random_state=42, penalty='l2', solver='lbfgs', max_iter=1000
    ),
    'Rf': RandomForestClassifier(
        class_weight='balanced', random_state=42, n_estimators=100
    ),
    'Knn': KNeighborsClassifier(
        n_neighbors=3 , weights='distance', algorithm='auto', leaf_size=30, p=2, metric='minkowski', n_jobs=-1
    ),
    'Xgb': XGBClassifier(
        random_state=42, n_estimators=100, use_label_encoder=False,
    ),
    'Gnb': GaussianNB()
}
for name, model in models.items():
    model.fit(X_train_selected, Y_train)
    score = model.score(X_test_selected, Y_test)
    print(f"{name} Accuracy: {score:.4f}")
    print(classification_report(Y_test, model.predict(X_test_selected)))
    print(confusion_matrix(Y_test, model.predict(X_test_selected)))

c:\Users\MSI\ML-final-project\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


Lr Accuracy: 0.7000
              precision    recall  f1-score   support

           0       0.78      0.72      0.75        29
           1       0.79      0.66      0.72        67
           2       0.51      0.79      0.62        24

    accuracy                           0.70       120
   macro avg       0.69      0.72      0.70       120
weighted avg       0.73      0.70      0.71       120

[[21  8  0]
 [ 5 44 18]
 [ 1  4 19]]
Rf Accuracy: 0.6167
              precision    recall  f1-score   support

           0       0.69      0.62      0.65        29
           1       0.67      0.64      0.66        67
           2       0.43      0.54      0.48        24

    accuracy                           0.62       120
   macro avg       0.60      0.60      0.60       120
weighted avg       0.63      0.62      0.62       120

[[18 11  0]
 [ 7 43 17]
 [ 1 10 13]]
Knn Accuracy: 0.6667
              precision    recall  f1-score   support

           0       0.92      0.38      0.54     

c:\Users\MSI\ML-final-project\.venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [12:10:37] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:793: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [11]:
from sklearn.model_selection import GridSearchCV

# Random Forest
rf_params = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10],
    'max_features': ['sqrt', 'log2']
}
rf_grid = GridSearchCV(
    RandomForestClassifier(class_weight='balanced', random_state=42),
    rf_params, cv=5, scoring='accuracy', n_jobs=-1
)
rf_grid.fit(X_train, Y_train)
print("Best RF:", rf_grid.best_params_, rf_grid.best_score_)

# KNN
knn_params = {'n_neighbors': [3,5,7,9,11], 'weights': ['uniform','distance'], 'metric': ['euclidean','manhattan']}
knn_grid = GridSearchCV(KNeighborsClassifier(), knn_params, cv=5, scoring='accuracy')
knn_grid.fit(X_train, Y_train)

# Logistic Regression
lr_params = {'C': [0.01,0.1,1,10,100], 'solver': ['lbfgs','saga'], 'max_iter': [500,1000]}
lr_grid = GridSearchCV(LogisticRegression(class_weight='balanced', random_state=42), lr_params, cv=5)
lr_grid.fit(X_train, Y_train)

# XGBoost
xgb_params = {
    'n_estimators': [100,200],
    'max_depth': [3,5,7],
    'learning_rate': [0.01,0.1,0.3],
    'subsample': [0.8,1.0]
}
xgb_grid = GridSearchCV(XGBClassifier(random_state=42, eval_metric='mlogloss'), xgb_params, cv=5)
xgb_grid.fit(X_train, Y_train)

Best RF: {'max_depth': None, 'max_features': 'sqrt', 'min_samples_split': 2, 'n_estimators': 200} 0.6932456140350878


c:\Users\MSI\ML-final-project\.venv\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Users\MSI\ML-final-project\.venv\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Users\MSI\ML-final-project\.venv\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Users\MSI\ML-final-project\.venv\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Users\MSI\ML-final-project\.venv\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Users\MSI\ML-final-project\.venv\Lib\site-packages\s

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","XGBClassifier...ree=None, ...)"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'learning_rate': [0.01, 0.1, ...], 'max_depth': [3, 5, ...], 'n_estimators': [100, 200], 'subsample': [0.8, 1.0]}"
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",None
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"verbose ver

Overfitting 